# From Unstructured Text to Interactive Knowledge Graphs Using LLMs

**Goal:** This notebook demonstrates a **highly granular, step-by-step process** to transform raw, unstructured text into a structured, interactive knowledge graph using Large Language Models (LLMs). We will extract factual information (SPO triples) and visualize the data transformations and final graph **directly within the notebook** at multiple stages.

**Target Audience:** Beginners to Intermediate Python users interested in NLP, Knowledge Graphs, and LLMs, who want to see the data evolve at each step.

**Approach:** We will break down the process into very small, logical steps. Each step will aim to perform a distinct action, followed by an output or visualization to show the immediate result. We'll use basic Python constructs and popular libraries, prioritizing clarity and step-by-step understanding over code conciseness.

## Theory: What is a Knowledge Graph?

A Knowledge Graph (KG) is a way to represent information as a network of entities and their relationships. Think of it like a structured database, but instead of tables, you have:

*   **Nodes (or Entities):** These represent real-world objects, concepts, people, places, organizations, etc. (e.g., 'Marie Curie', 'Physics', 'Paris'). In our graph, each unique subject or object from our extracted facts will become a node.
*   **Edges (or Relationships):** These represent the connections or interactions between entities. They typically have a direction and a label describing the relationship (e.g., 'Marie Curie' -- `won` --> 'Nobel Prize', 'Radium' -- `is element discovered by` --> 'Marie Curie'). In our graph, each predicate from our extracted facts defines an edge between the corresponding subject and object nodes.

Knowledge graphs make it easier to understand complex connections, infer new information, and query data in intuitive ways. Visualizing the graph helps immensely in spotting patterns and understanding the overall structure.

## Theory: Subject-Predicate-Object (SPO) Triples

The fundamental building block of many knowledge graphs derived from text is the **Subject-Predicate-Object (SPO)** triple. It's a simple structure that captures a single fact:

*   **Subject:** The entity the statement is about (becomes a node).
*   **Predicate:** The relationship or action connecting the subject and object (becomes the label on an edge).
*   **Object:** The entity related to the subject via the predicate (becomes another node).

**Example:** "Marie Curie discovered Radium" -> (`Marie Curie`, `discovered`, `Radium`).

This translates to graph nodes and edges: `(Marie Curie) -[discovered]-> (Radium)`.

LLMs help identify these triples by understanding language context.

## Step 1: Setup - Installing Libraries

First, we install the necessary Python libraries. We'll use:
*   `openai`: For LLM API interaction.
*   `networkx`: For graph data structures.
*   `ipycytoscape`: For interactive in-notebook graph visualization.
*   `ipywidgets`: Required by `ipycytoscape`.
*   `pandas`: For displaying data nicely in tables.

**Note:** You might need to restart the runtime/kernel after installation. Enable `ipywidgets` extension in classic Jupyter Notebook if needed.

In [31]:
# Install libraries (run this cell once)
%uv sync

# If in classic Jupyter Notebook (not Lab), you might need to enable the widget extension:
# jupyter nbextension enable --py widgetsnbextension

# --- IMPORTANT: Restart the kernel/runtime after running this cell! ---

Note: you may need to restart the kernel to use updated packages.


Resolved 225 packages in 1.68s
Audited 218 packages in 1ms


## Step 2: Setup - Importing Libraries

Now that the libraries are installed, we import the necessary components into our Python environment.

In [32]:
import openai  # For LLM interaction
import json  # For parsing LLM responses
import networkx as nx  # For creating and managing the graph data structure
import ipycytoscape  # For interactive in-notebook graph visualization
import pandas as pd  # For displaying data in tables
import os  # For accessing environment variables (safer for API keys)
import re  # For basic text cleaning (regular expressions)
import warnings  # To suppress potential deprecation warnings

# Configure settings for better display and fewer warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
pd.set_option("display.max_rows", 100)  # Show more rows in pandas tables
pd.set_option("display.max_colwidth", 150)  # Show more text width in pandas tables

print("Libraries imported successfully.")

Libraries imported successfully.


## Step 3: Configure LLM Access

We need to specify how to connect to the Large Language Model. This involves the API endpoint (URL) and the API key.

**IMPORTANT SECURITY NOTE:** Use environment variables or a secure secrets manager for API keys. **Do not hardcode keys directly in the notebook.**

**Environment Variable Setup (Example - run in your terminal *before* starting Jupyter):**
```bash
# For OpenAI
export OPENAI_API_KEY='your_openai_api_key'

# For Ollama (example)
export OPENAI_API_KEY='ollama' # Or any non-empty string
export OPENAI_API_BASE='http://localhost:11434/v1'

# For Nebius AI (example)
export OPENAI_API_KEY='your_nebius_api_key'
export OPENAI_API_BASE='https://api.studio.nebius.com/v1/'
```
First, we'll define the model name we intend to use.

In [ ]:
# --- Retrieve Credentials ---
from dotenv import load_dotenv

# 加载 .env 文件
load_dotenv()
api_key = os.getenv("LLM_API_KEY")
base_url = os.getenv(
    "LLM_BASE_URL"
)  # Will be None if not set (e.g., for standard OpenAI)
llm_model_name = os.getenv("KG_LLM_MODEL_NAME")

print(f"Intended LLM model: {llm_model_name}")

Intended LLM model: gemini-1.5-flash


Now, let's retrieve the API key and base URL from environment variables.

In [34]:
# --- FOR TESTING ONLY (Less Secure - Replace with Environment Variables) ---
# Uncomment and set these lines ONLY if you cannot set environment variables easily.
# api_key = "YOUR_API_KEY_HERE"  # <--- PASTE KEY HERE FOR TESTING ONLY
# base_url = "YOUR_API_BASE_URL_HERE" # <--- PASTE BASE URL HERE (if needed)
# Example for Nebius:
# base_url="https://api.studio.nebius.com/v1/"
# api_key="YOUR_NEBIUS_KEY"

print(f"Retrieved API Key: {'Set' if api_key else 'Not Set'}")
print(
    f"Retrieved Base URL: {base_url if base_url else 'Not Set (will use default OpenAI)'}"
)

Retrieved API Key: Set
Retrieved Base URL: https://xiaoai.plus/v1


Next, we validate the API key and initialize the `openai` client.

In [35]:
# --- Validate Key and Initialize Client ---
if not api_key:
    print(
        "Error: OPENAI_API_KEY environment variable not set or key not provided directly."
    )
    print(
        "Please set the environment variable (or uncomment/edit the test lines) and restart the kernel."
    )
    raise SystemExit("API Key configuration failed.")
else:
    try:
        client = openai.OpenAI(
            base_url=base_url,  # Pass None if not set, client handles default
            api_key=api_key,
        )
        print("OpenAI client initialized successfully.")
    except Exception as e:
        print(f"Error initializing OpenAI client: {e}")
        print("Check your API key, base URL (if used), and network connection.")
        raise SystemExit("LLM client initialization failed.")

OpenAI client initialized successfully.


Finally, let's define other LLM parameters like temperature and max tokens.

In [36]:
# --- Define LLM Call Parameters ---
llm_temperature = 0.0  # Lower temperature for more deterministic, factual output. 0.0 is best for extraction.
llm_max_tokens = 4096  # Max tokens for the LLM response (adjust based on model limits)

print(f"LLM Temperature set to: {llm_temperature}")
print(f"LLM Max Tokens set to: {llm_max_tokens}")

LLM Temperature set to: 0.0
LLM Max Tokens set to: 4096


## Step 4: Define Input Text

Here, we define the raw, unstructured text we want to process. We'll use the Marie Curie biography.

In [37]:
import os


def read_md_files_to_string(directory):
    """
    Recursively reads all .md files in the given directory and its subdirectories,
    concatenates their content into a single string, and prefixes each file's content
    with its filename.

    Args:
        directory (str): The root directory to start reading .md files.

    Returns:
        str: A single string containing the content of all .md files.
    """
    combined_text = ""

    for root, _, files in os.walk(directory):
        for file in files:
            print(file)
            if file.endswith(".md") or file.endswith(".txt"):
                file_path = os.path.join(root, file)
                try:
                    with open(file_path, "r", encoding="utf-8") as f:
                        file_content = f.read()
                        combined_text += f"--- File: {file} ---\n{file_content}\n\n"
                except Exception as e:
                    print(f"Error reading file {file_path}: {e}")

    return combined_text


# Example usage
directory_path = "../../data/local/"  # Replace with your target directory
unstructured_text = read_md_files_to_string(directory_path)
print("All .md files have been read and combined into unstructured_text.")
print(unstructured_text)

animal.txt
.meta.json
.meta.json
.meta.json
.meta.json
.meta.json
.meta.json
.meta.json
.meta.json
.meta.json
.meta.json
.meta.json
cat.txt
dog.txt
fox.txt
ham.txt
owl.txt
parrot.txt
wolf.txt
All .md files have been read and combined into unstructured_text.
--- File: animal.txt ---
动物是地球上生物多样性的重要组成部分。它们与植物、微生物共同构成了复杂的生态系统，维持着地球的生态平衡。在我看来，动物不仅仅是自然界的一部分，它们还有着丰富的内涵和意义。

首先，动物是大自然的奇迹。地球上的动物种类繁多，从微小的昆虫到巨大的蓝鲸，每一种动物都有其独特的生存方式和适应策略。昆虫以其惊人的数量和多样性占据了地球的每一个角落，而大象等大型动物则以其体型和力量令人惊叹。每种动物都通过进化发展出适应其生存环境的独特特征，这显示了生命的无比奇妙和顽强。

其次，动物与人类有着密切的关系。很多动物与人类共同生活，扮演着朋友、助手或工作伙伴的角色。宠物如猫狗，给予了人类无尽的陪伴和情感支持；而一些工作动物如导盲犬、缉毒犬等，帮助人类完成某些特殊任务。同时，动物在农业和科研中也扮演着重要角色，家畜和家禽为人类提供食物来源，而实验动物则是科学研究的重要对象。

然而，人类活动对动物的影响不容忽视。工业化进程、城市化扩张、森林砍伐等都威胁着动物的栖息地，导致许多物种濒临灭绝。偷猎和非法贸易则进一步加剧了某些物种的生存危机。为此，各国政府和国际组织不断加强动物保护法律法规，推动拯救濒危动物的努力。

此外，动物在文化和精神层面也占有重要地位。许多文化中，动物被视为神灵的化身或图腾，承载着丰厚的历史和信仰。在文学作品、电影和艺术中，动物角色时常被用来表现人类无法用语言表达的情感和哲理。

总之，动物是自然界的珍宝，也是人类的朋友和老师。保护动物及其生存环境，不仅是在保护我们的生态系统，也是在保护人类自身的未来。通过尊重和理解动物，我们能够与自然和谐相处，共同迎接更美好的未来。

--- File: cat.txt --

Let's display the input text and some basic statistics about it.

In [38]:
print("--- Input Text Loaded ---")
print(unstructured_text)
print("-" * 25)
# Basic stats visualization
char_count = len(unstructured_text)
word_count = len(unstructured_text.split())
print(f"Total characters: {char_count}")
print(f"Approximate word count: {word_count}")
print("-" * 25)

--- Input Text Loaded ---
--- File: animal.txt ---
动物是地球上生物多样性的重要组成部分。它们与植物、微生物共同构成了复杂的生态系统，维持着地球的生态平衡。在我看来，动物不仅仅是自然界的一部分，它们还有着丰富的内涵和意义。

首先，动物是大自然的奇迹。地球上的动物种类繁多，从微小的昆虫到巨大的蓝鲸，每一种动物都有其独特的生存方式和适应策略。昆虫以其惊人的数量和多样性占据了地球的每一个角落，而大象等大型动物则以其体型和力量令人惊叹。每种动物都通过进化发展出适应其生存环境的独特特征，这显示了生命的无比奇妙和顽强。

其次，动物与人类有着密切的关系。很多动物与人类共同生活，扮演着朋友、助手或工作伙伴的角色。宠物如猫狗，给予了人类无尽的陪伴和情感支持；而一些工作动物如导盲犬、缉毒犬等，帮助人类完成某些特殊任务。同时，动物在农业和科研中也扮演着重要角色，家畜和家禽为人类提供食物来源，而实验动物则是科学研究的重要对象。

然而，人类活动对动物的影响不容忽视。工业化进程、城市化扩张、森林砍伐等都威胁着动物的栖息地，导致许多物种濒临灭绝。偷猎和非法贸易则进一步加剧了某些物种的生存危机。为此，各国政府和国际组织不断加强动物保护法律法规，推动拯救濒危动物的努力。

此外，动物在文化和精神层面也占有重要地位。许多文化中，动物被视为神灵的化身或图腾，承载着丰厚的历史和信仰。在文学作品、电影和艺术中，动物角色时常被用来表现人类无法用语言表达的情感和哲理。

总之，动物是自然界的珍宝，也是人类的朋友和老师。保护动物及其生存环境，不仅是在保护我们的生态系统，也是在保护人类自身的未来。通过尊重和理解动物，我们能够与自然和谐相处，共同迎接更美好的未来。

--- File: cat.txt ---
猫是一种常见的宠物，以其温顺和独立性闻名。它们通常有柔软的毛发、敏锐的感官以及敏捷的动作。猫可以很好地适应各种居住环境，它们的呼噜声常被认为具有舒缓和疗愈作用。

--- File: dog.txt ---
狗是人类的宠物，是人类最好的朋友之一，忠诚、友好且常常充满活力。它们通常能够提供陪伴、保护家庭，并且可以通过训练来进行各种工作，如导盲、救援和警戒。狗有很多品种，每种都有自己独特的特性和需求。无论是作为家庭宠物还是工作伙伴，狗都在人们的生活中扮演着重要

## Step 5: Text Chunking (Optional but Recommended)

LLMs have context limits. For longer texts, we need to break them into smaller chunks. We'll define the chunk size and overlap.

*   **Chunk Size:** Max words per chunk.
*   **Overlap:** Words shared between consecutive chunks to preserve context.

In [39]:
# --- Chunking Configuration ---
chunk_size = 300  # Number of words per chunk (adjust as needed)
overlap = 30  # Number of words to overlap (must be < chunk_size)

print(f"Chunk Size set to: {chunk_size} words")
print(f"Overlap set to: {overlap} words")

# --- Basic Validation ---
if overlap >= chunk_size and chunk_size > 0:
    print(f"Error: Overlap ({overlap}) must be smaller than chunk size ({chunk_size}).")
    raise SystemExit("Chunking configuration error.")
else:
    print("Chunking configuration is valid.")

Chunk Size set to: 300 words
Overlap set to: 30 words
Chunking configuration is valid.


First, let's split the input text into a list of words.

In [40]:
import jieba

words = list(jieba.cut(unstructured_text))
total_words = len(words)

print(f"Text split into {total_words} words.")
# Visualize the first 20 words
print(f"First 20 words: {words[:20]}")

Text split into 2474 words.
First 20 words: ['---', ' ', 'File', ':', ' ', 'animal', '.', 'txt', ' ', '---', '\n', '动物', '是', '地球', '上', '生物', '多样性', '的', '重要', '组成部分']


Now, we'll perform the chunking based on the configuration.

In [41]:
chunks = []
start_index = 0
chunk_number = 1

print("Starting chunking process...")

while start_index < total_words:
    end_index = min(start_index + chunk_size, total_words)
    chunk_text = " ".join(words[start_index:end_index])
    chunks.append({"text": chunk_text, "chunk_number": chunk_number})

    # print(f"  Created chunk {chunk_number}: words {start_index} to {end_index-1}") # Uncomment for detailed log

    # Calculate the start of the next chunk
    next_start_index = start_index + chunk_size - overlap

    # Ensure progress is made
    if next_start_index <= start_index:
        if end_index == total_words:
            break  # Already processed the last part
        next_start_index = start_index + 1

    start_index = next_start_index
    chunk_number += 1

    # Safety break (optional)
    if chunk_number > total_words:  # Simple safety
        print("Warning: Chunking loop exceeded total word count, breaking.")
        break

print(f"\nText successfully split into {len(chunks)} chunks.")

Starting chunking process...

Text successfully split into 10 chunks.


Let's visualize the created chunks using Pandas DataFrame.

In [42]:
print("--- Chunk Details ---")
if chunks:
    # Create a DataFrame for better visualization
    chunks_df = pd.DataFrame(chunks)
    chunks_df["word_count"] = chunks_df["text"].apply(lambda x: len(x.split()))
    display(chunks_df[["chunk_number", "word_count", "text"]])
else:
    print("No chunks were created (text might be shorter than chunk size).")
print("-" * 25)

--- Chunk Details ---


,chunk_number,word_count,text
0,1,290,--- File : animal . txt --- \n 动物 是 地球 上 生物 多样性 的 重要 组成部分 。 它们 与 植物 、 微生物 共同 构成 了 复杂 的 生态系统 ， 维持 着 地球 的 生态平衡 。 在我看来 ， 动物 不仅仅 是 自然界 的 一部分 ， 它...
1,2,284,导致 许多 物种 濒临灭绝 。 偷猎 和 非法 贸易 则 进一步 加剧 了 某些 物种 的 生存 危机 。 为此 ， 各国 政府 和 国际 组织 不断加强 动物 保护 法律法规 ， 推动 拯救 濒危动物 的 努力 。 \n \n 此外 ， 动物 在 文化 和 精神 层面 也 占有 重要 地位...
2,3,288,， 如导盲 、 救援 和 警戒 。 狗 有 很多 品种 ， 每种 都 有 自己 独特 的 特性 和 需求 。 无论是 作为 家庭 宠物 还是 工作 伙伴 ， 狗 都 在 人们 的 生活 中 扮演着 重要 角色 。 \n \n --- File : fox . txt --- \n...
3,4,294,狐狸 常 被 赋予 不同 的 角色 。 例如 ， 狐狸 在 中国 民间传说 中 有时 被 看作 是 善于 变化 和 狡黠 的 化身 。 类似 地 ， 在 西方 故事 中 ， 狐狸 也 常 被 刻画 成 一种 聪明 的 形象 ， 能够 运用 智谋 解决 麻烦 。 在 许多 寓言 和 故事 中 ...
4,5,288,愈发 丰富 和 立体 。 狐狸 不仅 是 自然界 的 精灵 ， 也 成为 激发 人们 创造 灵感 的 源泉 。 狐狸 的 故事 和 传说 将 继续 流传 ， 成为 人类 文化 的 一部分 。 这种 生物 的 魅力 深入人心 ， 不论是 在 文学作品 中 ， 还是 在 日常生活 的 观念 里 ...
5,6,290,、 坚果 、 谷物 等 ， 偶尔 也 会 吃 一些 昆虫 和 小型 动物 。 这种 饮食习惯 使 它们 能够 有效 储存 食物 ， 以 应对 食物 匮乏 的 时刻 。 仓鼠 天生 具备 将 食物 储存 在 颊囊 中 的 能力 ， 在 觅食 时 常常 把 食物 塞满 颊囊 ， 然后 带回 巢穴...
6,7,280,因为 照料 小 动物 是 一个 长期 的 承诺 ， 需要 用心 投入 。 \n \n 总之 ， 仓鼠 是 一种 可爱 的 动物 ， 适合 家庭 饲养 ， 但 前提 是 要 了解 它们 的 习性 ， 并 确保 提供 适合 的 居住 环境 和 饮食 条件 。 只有 这样 ， 才 能够 让 仓鼠 ...
7,8,288,健康 和 快乐 。 \n \n --- File : wolf . txt --- \n 狼 是 一种 极具 智慧 和 社会性 的 动物 ， 属于 犬科 ， 是 一种 野生 哺乳动物 。 狼 通常 生活 在 群体 中 ， 以 群体 合作 的 方式 进行 猎食 ， 这种 社会 组织...
8,9,294,一次 。 狼群 一般 只 允许 一个 主导 的 交配 对 进行 繁殖 ， 这 确保 了 群体 资源 能够 充分 支持 幼狼 的 成长 。 幼狼 诞生 后 ， 整个 狼群 都 会 参与 照料 和 保护 幼狼 ， 直到 它们 能够 独立 生存 。 狼 的 幼崽 通常 在 约 六个月 大时 开始 ...
9,10,42,享受 和 利用 自然资源 的 同时 ， 保护 狼 以及 它们 的 栖息地 变得 尤为重要 。 这 不仅 能 维持 生态系统 的 健康 发展 ， 也 能 确保 自然界 中 这种 神秘 而 迷人 的 动物 能够 继续 发挥 其 作用 。 \n \n


-------------------------


## Step 6: Define the LLM Prompt for Extraction

This is a critical step. We need to carefully instruct the LLM to extract SPO triples in a specific JSON format. We'll define a system prompt (role) and a user prompt template (instructions).

**Key Instructions Emphasized:**
*   Extract `Subject-Predicate-Object` triples.
*   Output *only* a valid JSON array of objects.
*   Each object must have `"subject"`, `"predicate"`, `"object"` keys.
*   Predicates should be concise (1-3 words).
*   All output values must be lowercase.
*   Resolve pronouns to specific entity names.
*   No extra text, explanations, or markdown code fences around the JSON.

In [43]:
# --- System Prompt: Sets the context/role for the LLM ---
extraction_system_prompt = """
你是一位处理知识图谱提取的 AI 专家。我将用英文给出 Prompt，但你需要对处理文本中的语言保持原样，即不必翻译为英文后输出。
You are an AI expert specialized in knowledge graph extraction. 
Your task is to identify and extract factual Subject-Predicate-Object (SPO) triples from the given text.
Focus on accuracy and adhere strictly to the JSON output format requested in the user prompt.
Extract core entities and the most direct relationship.
"""

# --- User Prompt Template: Contains specific instructions and the text ---
extraction_user_prompt_template = """
Please extract Subject-Predicate-Object (S-P-O) triples from the text below. 对于文本中的名词、关系，如果原文使用中文表述，你生成的内容也应该为中文；即：不必将内容翻译为英文再输出。

如果节点和关系中出现用 $$ 括起来的公式，你需要将公式表达为不含 $$ 的文本形式，不使用 latex 语法，不必严谨表述，只要能大致看出公式形式即可。

**VERY IMPORTANT RULES:**
1.  **Output Format:** Respond ONLY with a single, valid JSON array. Each element MUST be an object with keys "subject", "predicate", "object".
2.  **JSON Only:** Do NOT include any text before or after the JSON array (e.g., no 'Here is the JSON:' or explanations). Do NOT use markdown ```json ... ``` tags.
3.  **Concise Predicates:** Keep the 'predicate' value concise (1-3 words, ideally 1-2). Use verbs or short verb phrases (e.g., 'discovered', 'was born in', 'won').
4.  **Lowercase:** ALL values for 'subject', 'predicate', and 'object' MUST be lowercase.
5.  **Pronoun Resolution:** Replace pronouns (she, he, it, her, etc.) with the specific lowercase entity name they refer to based on the text context (e.g., 'marie curie').
6.  **Specificity:** Capture specific details (e.g., 'nobel prize in physics' instead of just 'nobel prize' if specified).
7.  **Completeness:** Extract all distinct factual relationships mentioned.

**Text to Process:**
```text
{text_chunk}
```

**Required JSON Output Format Example:**
[
  {{ "subject": "marie curie", "predicate": "discovered", "object": "radium" }},
  {{ "subject": "marie curie", "predicate": "won", "object": "nobel prize in physics" }}
]

**Your JSON Output (MUST start with '[' and end with ']'):**
"""

Let's display the prompts we've defined to verify them.

In [44]:
print("--- System Prompt ---")
print(extraction_system_prompt)
print("\n" + "-" * 25 + "\n")

print("--- User Prompt Template (Structure) ---")
# Show structure, replacing the placeholder for clarity
print(
    extraction_user_prompt_template.replace(
        "{text_chunk}", "[... text chunk goes here ...]"
    )
)
print("\n" + "-" * 25 + "\n")

# Show an example of the *actual* prompt that will be sent for the first chunk
print("--- Example Filled User Prompt (for Chunk 1) ---")
if chunks:
    example_filled_prompt = extraction_user_prompt_template.format(
        text_chunk=chunks[0]["text"]
    )
    # Displaying a limited portion for brevity
    print(
        example_filled_prompt[:600]
        + "\n[... rest of chunk text ...]\n"
        + example_filled_prompt[-200:]
    )
else:
    print("No chunks available to create an example filled prompt.")
print("\n" + "-" * 25)

--- System Prompt ---

你是一位处理知识图谱提取的 AI 专家。我将用英文给出 Prompt，但你需要对处理文本中的语言保持原样，即不必翻译为英文后输出。
You are an AI expert specialized in knowledge graph extraction. 
Your task is to identify and extract factual Subject-Predicate-Object (SPO) triples from the given text.
Focus on accuracy and adhere strictly to the JSON output format requested in the user prompt.
Extract core entities and the most direct relationship.


-------------------------

--- User Prompt Template (Structure) ---

Please extract Subject-Predicate-Object (S-P-O) triples from the text below. 对于文本中的名词、关系，如果原文使用中文表述，你生成的内容也应该为中文；即：不必将内容翻译为英文再输出。

如果节点和关系中出现用 $$ 括起来的公式，你需要将公式表达为不含 $$ 的文本形式，不使用 latex 语法，不必严谨表述，只要能大致看出公式形式即可。

**VERY IMPORTANT RULES:**
1.  **Output Format:** Respond ONLY with a single, valid JSON array. Each element MUST be an object with keys "subject", "predicate", "object".
2.  **JSON Only:** Do NOT include any text before or after the JSON array (e.g., no 'Here is the JSON:' or explanations). Do NOT use markdow

## Step 7: LLM Interaction - Extracting Triples (Chunk by Chunk)

Now we loop through each text chunk, send it to the LLM with our prompts, and attempt to parse the expected JSON output. We will show the process for each chunk.

In [45]:
# Initialize lists to store results and failures
all_extracted_triples = []
failed_chunks = []

print(
    f"Starting triple extraction from {len(chunks)} chunks using model '{llm_model_name}'..."
)
# We will process chunks one by one in the following cells.

Starting triple extraction from 10 chunks using model 'gemini-1.5-flash'...


### Processing Chunk 1 (Example - loop structure will handle all)

In [46]:
# --- This cell represents the core logic inside the loop for ONE chunk ---
# --- In a real run, this logic would be in a loop like the original notebook ---
# --- We show it step-by-step for the first chunk for clarity ---

# chunk_index = 0 # For demonstration, we process only the first chunk here

print("model: ", llm_model_name)

for chunk_index in range(len(chunks)):
    chunk_info = chunks[chunk_index]
    chunk_text = chunk_info["text"]
    chunk_num = chunk_info["chunk_number"]

    print(f"\n--- Processing Chunk {chunk_num}/{len(chunks)} --- ")

    # 1. Format the User Prompt
    print("1. Formatting User Prompt...")
    user_prompt = extraction_user_prompt_template.format(text_chunk=chunk_text)
    # print(f"   Formatted Prompt (Snippet): {user_prompt[:200]}...{user_prompt[-100:]}") # Optional: View prompt

    llm_output = None
    error_message = None

    try:
        # 2. Make the API Call
        print("2. Sending request to LLM...")
        response = client.chat.completions.create(
            model=llm_model_name,
            messages=[
                {"role": "system", "content": extraction_system_prompt},
                {"role": "user", "content": user_prompt},
            ],
            temperature=llm_temperature,
            max_tokens=llm_max_tokens,
            # Request JSON output format - helps models that support it
            response_format={"type": "json_object"},
        )
        print("   LLM response received.")

        # 3. Extract Raw Response Content
        print("3. Extracting raw response content...")
        llm_output = response.choices[0].message.content.strip()
        print("--- Raw LLM Output (Chunk {chunk_num}) ---")
        print(llm_output)
        print("-" * 20)

    except Exception as e:
        error_message = str(e)
        print(f"   ERROR during API call: {error_message}")
        failed_chunks.append(
            {
                "chunk_number": chunk_num,
                "error": f"API/Processing Error: {error_message}",
                "response": "",
            }
        )

    # 4. Parse JSON (if API call succeeded)
    parsed_json = None
    parsing_error = None
    if llm_output is not None:
        print("4. Attempting to parse JSON from response...")
        try:
            # Strategy 1: Direct parsing (ideal)
            parsed_data = json.loads(llm_output)

            # Handle if response_format={'type':'json_object'} returns a dict containing the list
            if isinstance(parsed_data, dict):
                print("   Detected dictionary response, attempting to extract list...")
                list_values = [v for v in parsed_data.values() if isinstance(v, list)]
                if len(list_values) == 1:
                    parsed_json = list_values[0]
                    print("      Successfully extracted list from dictionary.")
                else:
                    raise ValueError(
                        "JSON object received, but doesn't contain a single list of triples."
                    )
            elif isinstance(parsed_data, list):
                parsed_json = parsed_data
                print("   Successfully parsed JSON list directly.")
            else:
                raise ValueError(
                    "Parsed JSON is not a list or expected dictionary wrapper."
                )

        except json.JSONDecodeError as json_err:
            parsing_error = f"JSONDecodeError: {json_err}. Trying regex fallback..."
            print(f"   {parsing_error}")
            # Strategy 2: Regex fallback for arrays potentially wrapped in text/markdown
            match = re.search(r"^\s*(\[.*?\])\s*$", llm_output, re.DOTALL)
            if match:
                json_string_extracted = match.group(1)
                print("      Regex found potential JSON array structure.")
                try:
                    parsed_json = json.loads(json_string_extracted)
                    print("      Successfully parsed JSON from regex extraction.")
                    parsing_error = None  # Clear previous error
                except json.JSONDecodeError as nested_err:
                    parsing_error = f"JSONDecodeError after regex: {nested_err}"
                    print(f"      ERROR: Regex content is not valid JSON: {nested_err}")
            else:
                parsing_error = "JSONDecodeError and Regex fallback failed."
                print("      ERROR: Regex could not find JSON array structure.")

        except ValueError as val_err:
            parsing_error = (
                f"ValueError: {val_err}"  # Catches issues with unexpected structure
            )
            print(f"   ERROR: {parsing_error}")

        # --- Show Parsed Result (or error) ---
        if parsed_json is not None:
            print("--- Parsed JSON Data (Chunk {chunk_num}) ---")
            print(json.dumps(parsed_json, indent=2))  # Pretty print the JSON
            print("-" * 20)
        else:
            print(f"--- JSON Parsing FAILED (Chunk {chunk_num}) --- ")
            print(f"   Final Parsing Error: {parsing_error}")
            print("-" * 20)
            failed_chunks.append(
                {
                    "chunk_number": chunk_num,
                    "error": f"Parsing Failed: {parsing_error}",
                    "response": llm_output,
                }
            )

    # 5. Validate and Store Triples (if parsing succeeded)
    if parsed_json is not None:
        print("5. Validating structure and extracting triples...")
        valid_triples_in_chunk = []
        invalid_entries = []
        if isinstance(parsed_json, list):
            for item in parsed_json:
                if isinstance(item, dict) and all(
                    k in item for k in ["subject", "predicate", "object"]
                ):
                    # Basic check: ensure values are strings (can be refined)
                    if all(
                        isinstance(item[k], str)
                        for k in ["subject", "predicate", "object"]
                    ):
                        item["chunk"] = chunk_num  # Add source chunk info
                        valid_triples_in_chunk.append(item)
                    else:
                        invalid_entries.append(
                            {"item": item, "reason": "Non-string value"}
                        )
                else:
                    invalid_entries.append(
                        {"item": item, "reason": "Incorrect structure/keys"}
                    )
        else:
            print("   ERROR: Parsed data is not a list, cannot extract triples.")
            invalid_entries.append({"item": parsed_json, "reason": "Not a list"})
            # Also add to failed chunks if the overall structure was wrong
            if not any(fc["chunk_number"] == chunk_num for fc in failed_chunks):
                failed_chunks.append(
                    {
                        "chunk_number": chunk_num,
                        "error": "Parsed data not a list",
                        "response": llm_output,
                    }
                )

        # --- Show Validation Results ---
        print(f"   Found {len(valid_triples_in_chunk)} valid triples in this chunk.")
        if invalid_entries:
            print(f"   Skipped {len(invalid_entries)} invalid entries.")
            # print(f"   Invalid entries details: {invalid_entries}") # Uncomment for debugging

        # --- Display Valid Triples from this Chunk ---
        if valid_triples_in_chunk:
            print(f"--- Valid Triples Extracted (Chunk {chunk_num}) ---")
            display(pd.DataFrame(valid_triples_in_chunk))
            print("-" * 20)
            # Add to the main list
            all_extracted_triples.extend(valid_triples_in_chunk)
        else:
            print("--- No valid triples extracted from this chunk. ---")
            print("-" * 20)

    # --- Update Running Total (Visual Feedback) ---
    print(f"--- Running Total Triples Extracted: {len(all_extracted_triples)} --- ")
    print(f"--- Failed Chunks So Far: {len(failed_chunks)} --- ")

print("\nFinished processing this chunk.")
# --- IMPORTANT: In a full run, you would uncomment the loop in the original notebook ---
# --- and remove the `chunk_index = 0` line to process ALL chunks. ---

model:  gemini-1.5-flash

--- Processing Chunk 1/10 --- 
1. Formatting User Prompt...
2. Sending request to LLM...
   LLM response received.
3. Extracting raw response content...
--- Raw LLM Output (Chunk {chunk_num}) ---
[{"subject": "动物", "predicate": "是", "object": "地球上生物多样性的重要组成部分"}, {"subject": "动物", "predicate": "与", "object": "植物、微生物共同构成复杂的生态系统"}, {"subject": "动物", "predicate": "维持着", "object": "地球的生态平衡"}, {"subject": "动物", "predicate": "是", "object": "大自然奇迹"}, {"subject": "地球上的动物", "predicate": "种类", "object": "繁多"}, {"subject": "昆虫", "predicate": "占据了", "object": "地球的每一个角落"}, {"subject": "大象等大型动物", "predicate": "以", "object": "体型和力量令人惊叹"}, {"subject": "每种动物", "predicate": "通过进化发展出", "object": "适应其生存环境的独特特征"}, {"subject": "动物", "predicate": "与", "object": "人类有着密切的关系"}, {"subject": "很多动物", "predicate": "扮演着", "object": "朋友、助手或工作伙伴的角色"}, {"subject": "宠物", "predicate": "给予", "object": "人类无尽的陪伴和情感支持"}, {"subject": "一些工作动物", "predicate": "帮助", "object": "人类完成某些特殊任务"}, {"subject": "动

,subject,predicate,object,chunk
0,动物,是,地球上生物多样性的重要组成部分,1
1,动物,与,植物、微生物共同构成复杂的生态系统,1
2,动物,维持着,地球的生态平衡,1
3,动物,是,大自然奇迹,1
4,地球上的动物,种类,繁多,1
5,昆虫,占据了,地球的每一个角落,1
6,大象等大型动物,以,体型和力量令人惊叹,1
7,每种动物,通过进化发展出,适应其生存环境的独特特征,1
8,动物,与,人类有着密切的关系,1
9,很多动物,扮演着,朋友、助手或工作伙伴的角色,1


--------------------
--- Running Total Triples Extracted: 19 --- 
--- Failed Chunks So Far: 0 --- 

--- Processing Chunk 2/10 --- 
1. Formatting User Prompt...
2. Sending request to LLM...
   LLM response received.
3. Extracting raw response content...
--- Raw LLM Output (Chunk {chunk_num}) ---
[{"subject": "偷猎和非法贸易", "predicate": "加剧", "object": "某些物种的生存危机"}, {"subject": "各国政府和国际组织", "predicate": "加强", "object": "动物保护法律法规"}, {"subject": "各国政府和国际组织", "predicate": "推动", "object": "拯救濒危动物的努力"}, {"subject": "许多文化", "predicate": "视为", "object": "神灵的化身或图腾"}, {"subject": "动物", "predicate": "承载", "object": "丰厚的历史和信仰"}, {"subject": "动物角色", "predicate": "用来表现", "object": "人类无法用语言表达的情感和哲理"}, {"subject": "动物", "predicate": "是", "object": "自然界的珍宝"}, {"subject": "动物", "predicate": "是", "object": "人类的朋友和老师"}, {"subject": "保护动物及其生存环境", "predicate": "是", "object": "保护我们生态系统"}, {"subject": "保护动物及其生存环境", "predicate": "是", "object": "保护人类自身的未来"}, {"subject": "猫", "predicate": "是", "object": "一种常见的宠物"}, {

,subject,predicate,object,chunk
0,偷猎和非法贸易,加剧,某些物种的生存危机,2
1,各国政府和国际组织,加强,动物保护法律法规,2
2,各国政府和国际组织,推动,拯救濒危动物的努力,2
3,许多文化,视为,神灵的化身或图腾,2
4,动物,承载,丰厚的历史和信仰,2
5,动物角色,用来表现,人类无法用语言表达的情感和哲理,2
6,动物,是,自然界的珍宝,2
7,动物,是,人类的朋友和老师,2
8,保护动物及其生存环境,是,保护我们生态系统,2
9,保护动物及其生存环境,是,保护人类自身的未来,2


--------------------
--- Running Total Triples Extracted: 42 --- 
--- Failed Chunks So Far: 0 --- 

--- Processing Chunk 3/10 --- 
1. Formatting User Prompt...
2. Sending request to LLM...
   LLM response received.
3. Extracting raw response content...
--- Raw LLM Output (Chunk {chunk_num}) ---
[{"subject": "狗", "predicate": "扮演", "object": "重要角色"}, {"subject": "狐狸", "predicate": "属于", "object": "犬科动物"}, {"subject": "狐狸", "predicate": "闻名", "object": "聪明,狡猾"}, {"subject": "狐狸", "predicate": "具有", "object": "尖尖的鼻子和耳朵"}, {"subject": "狐狸", "predicate": "以", "object": "小型哺乳动物、鸟类以及昆虫为食"}, {"subject": "狐狸", "predicate": "吃", "object": "植物果实"}, {"subject": "狐狸", "predicate": "是", "object": "独居动物"}, {"subject": "狐狸", "predicate": "交往", "object": "其他狐狸"}, {"subject": "狐狸", "predicate": "建立", "object": "自己的领地"}, {"subject": "母狐狸", "predicate": "产仔", "object": "隐蔽的巢穴"}, {"subject": "狐狸", "predicate": "被看作", "object": "善于变化和狡黠的化身"}]
--------------------
4. Attempting to parse JSON from response...

...

--------------------
--- Running Total Triples Extracted: 175 --- 
--- Failed Chunks So Far: 0 --- 

--- Processing Chunk 9/10 --- 
1. Formatting User Prompt...
2. Sending request to LLM...
   LLM response received.
3. Extracting raw response content...
--- Raw LLM Output (Chunk {chunk_num}) ---
[{"subject": "狼群", "predicate": "允许", "object": "一个主导的交配对进行繁殖"}, {"subject": "整个狼群", "predicate": "参与", "object": "照料和保护幼狼"}, {"subject": "幼狼", "predicate": "跟随", "object": "成年狼一起猎食"}, {"subject": "人类活动", "predicate": "导致", "object": "狼的生存环境面临诸多挑战"}, {"subject": "人类", "predicate": "猎杀", "object": "狼"}, {"subject": "栖息地", "predicate": "破坏", "object": "狼"}, {"subject": "保护意识", "predicate": "增强", "object": "狼的数量"}, {"subject": "生态保护措施", "predicate": "实施", "object": "狼的数量"}, {"subject": "狼", "predicate": "被视为", "object": "勇敢和自由的象征"}, {"subject": "狼", "predicate": "被视为", "object": "危险和狡猾的化身"}, {"subject": "狼", "predicate": "用于", "object": "研究有关行为学和群体动力学"}, {"subject": "狼", "predicate": "扮演", "object

,subject,predicate,object,chunk
0,狼群,允许,一个主导的交配对进行繁殖,9
1,整个狼群,参与,照料和保护幼狼,9
2,幼狼,跟随,成年狼一起猎食,9
3,人类活动,导致,狼的生存环境面临诸多挑战,9
4,人类,猎杀,狼,9
5,栖息地,破坏,狼,9
6,保护意识,增强,狼的数量,9
7,生态保护措施,实施,狼的数量,9
8,狼,被视为,勇敢和自由的象征,9
9,狼,被视为,危险和狡猾的化身,9


--------------------
--- Running Total Triples Extracted: 188 --- 
--- Failed Chunks So Far: 0 --- 

--- Processing Chunk 10/10 --- 
1. Formatting User Prompt...
2. Sending request to LLM...
   LLM response received.
3. Extracting raw response content...
--- Raw LLM Output (Chunk {chunk_num}) ---
[{"subject": "保护狼以及它们的栖息地", "predicate": "变得", "object": "尤为重要"}, {"subject": "保护狼以及它们的栖息地", "predicate": "能维持", "object": "生态系统健康发展"}, {"subject": "保护狼以及它们的栖息地", "predicate": "能确保", "object": "自然界中这种神秘而迷人的动物能够继续发挥其作用"}]
--------------------
4. Attempting to parse JSON from response...
   Successfully parsed JSON list directly.
--- Parsed JSON Data (Chunk {chunk_num}) ---
[
  {
    "subject": "\u4fdd\u62a4\u72fc\u4ee5\u53ca\u5b83\u4eec\u7684\u6816\u606f\u5730",
    "predicate": "\u53d8\u5f97",
    "object": "\u5c24\u4e3a\u91cd\u8981"
  },
  {
    "subject": "\u4fdd\u62a4\u72fc\u4ee5\u53ca\u5b83\u4eec\u7684\u6816\u606f\u5730",
    "predicate": "\u80fd\u7ef4\u6301",
    "object": "\u751f\u6001\u

,subject,predicate,object,chunk
0,保护狼以及它们的栖息地,变得,尤为重要,10
1,保护狼以及它们的栖息地,能维持,生态系统健康发展,10
2,保护狼以及它们的栖息地,能确保,自然界中这种神秘而迷人的动物能够继续发挥其作用,10


--------------------
--- Running Total Triples Extracted: 191 --- 
--- Failed Chunks So Far: 0 --- 

Finished processing this chunk.


### Extraction Summary (After Processing All Chunks)

**(Note:** The previous cell only processed *one* chunk for demonstration. In a full run, the loop would process all chunks. The summary below reflects the state *after* the demonstrated single chunk processing. Run the full loop from the original notebook to get the complete results).**

Let's summarize the extraction results and display all accumulated triples.

In [47]:
# --- Summary of Extraction (Reflecting state after the single chunk demo) ---
print("\n--- Overall Extraction Summary ---")
print(f"Total chunks defined: {len(chunks)}")
processed_chunks = len(chunks) - len(
    failed_chunks
)  # Approximation if loop isn't run fully
print(
    f"Chunks processed (attempted): {processed_chunks + len(failed_chunks)}"
)  # Chunks we looped through
print(
    f"Total valid triples extracted across all processed chunks: {len(all_extracted_triples)}"
)
print(f"Number of chunks that failed API call or parsing: {len(failed_chunks)}")

if failed_chunks:
    print("\nDetails of Failed Chunks:")
    for failure in failed_chunks:
        print(f"  Chunk {failure['chunk_number']}: Error: {failure['error']}")
        # print(f"    Response (start): {failure.get('response', '')[:100]}...") # Uncomment for more detail
print("-" * 25)

# Display all extracted triples using Pandas
print("\n--- All Extracted Triples (Before Normalization) ---")
if all_extracted_triples:
    all_triples_df = pd.DataFrame(all_extracted_triples)
    display(all_triples_df)
else:
    print("No triples were successfully extracted.")
print("-" * 25)


--- Overall Extraction Summary ---
Total chunks defined: 10
Chunks processed (attempted): 10
Total valid triples extracted across all processed chunks: 191
Number of chunks that failed API call or parsing: 0
-------------------------

--- All Extracted Triples (Before Normalization) ---


,subject,predicate,object,chunk
0,动物,是,地球上生物多样性的重要组成部分,1
1,动物,与,植物、微生物共同构成复杂的生态系统,1
2,动物,维持着,地球的生态平衡,1
3,动物,是,大自然奇迹,1
4,地球上的动物,种类,繁多,1
...,...,...,...,...
186,狼,扮演,重要角色,9
187,狼,发挥,关键作用,9
188,保护狼以及它们的栖息地,变得,尤为重要,10
189,保护狼以及它们的栖息地,能维持,生态系统健康发展,10


-------------------------


## Step 8: Normalize and De-duplicate Triples

Now, we clean up the extracted triples:
1.  **Normalize:** Trim whitespace, convert to lowercase.
2.  **Filter:** Remove triples with empty parts after normalization.
3.  **De-duplicate:** Remove exact duplicate `(subject, predicate, object)` combinations.

In [48]:
# Initialize lists and tracking variables
normalized_triples = []
seen_triples = set()  # Tracks (subject, predicate, object) tuples
original_count = len(all_extracted_triples)
empty_removed_count = 0
duplicates_removed_count = 0

print(f"Starting normalization and de-duplication of {original_count} triples...")

Starting normalization and de-duplication of 191 triples...


We'll iterate through the extracted triples, clean them, and check for duplicates. We'll show the first few transformations.

In [49]:
print("Processing triples for normalization (showing first 5 examples):")
example_limit = 5
processed_count = 0

for i, triple in enumerate(all_extracted_triples):
    show_example = i < example_limit
    if show_example:
        print(f"\n--- Example {i + 1} ---")
        print(f"Original Triple (Chunk {triple.get('chunk', '?')}): {triple}")

    subject_raw = triple.get("subject")
    predicate_raw = triple.get("predicate")
    object_raw = triple.get("object")
    chunk_num = triple.get("chunk", "unknown")

    triple_valid = False
    normalized_sub, normalized_pred, normalized_obj = None, None, None

    if (
        isinstance(subject_raw, str)
        and isinstance(predicate_raw, str)
        and isinstance(object_raw, str)
    ):
        # 1. Normalize
        normalized_sub = subject_raw.strip().lower()
        normalized_pred = re.sub(r"\s+", " ", predicate_raw.strip().lower()).strip()
        normalized_obj = object_raw.strip().lower()
        if show_example:
            print(
                f"Normalized: SUB='{normalized_sub}', PRED='{normalized_pred}', OBJ='{normalized_obj}'"
            )

        # 2. Filter Empty
        if normalized_sub and normalized_pred and normalized_obj:
            triple_identifier = (normalized_sub, normalized_pred, normalized_obj)

            # 3. De-duplicate
            if triple_identifier not in seen_triples:
                normalized_triples.append(
                    {
                        "subject": normalized_sub,
                        "predicate": normalized_pred,
                        "object": normalized_obj,
                        "source_chunk": chunk_num,
                    }
                )
                seen_triples.add(triple_identifier)
                triple_valid = True
                if show_example:
                    print("Status: Kept (New Unique Triple)")
            else:
                duplicates_removed_count += 1
                if show_example:
                    print("Status: Discarded (Duplicate)")
        else:
            empty_removed_count += 1
            if show_example:
                print("Status: Discarded (Empty component after normalization)")
    else:
        empty_removed_count += 1  # Count non-string/missing as needing removal
        if show_example:
            print("Status: Discarded (Non-string or missing component)")
    processed_count += 1

print(f"\n... Finished processing {processed_count} triples.")

Processing triples for normalization (showing first 5 examples):

--- Example 1 ---
Original Triple (Chunk 1): {'subject': '动物', 'predicate': '是', 'object': '地球上生物多样性的重要组成部分', 'chunk': 1}
Normalized: SUB='动物', PRED='是', OBJ='地球上生物多样性的重要组成部分'
Status: Kept (New Unique Triple)

--- Example 2 ---
Original Triple (Chunk 1): {'subject': '动物', 'predicate': '与', 'object': '植物、微生物共同构成复杂的生态系统', 'chunk': 1}
Normalized: SUB='动物', PRED='与', OBJ='植物、微生物共同构成复杂的生态系统'
Status: Kept (New Unique Triple)

--- Example 3 ---
Original Triple (Chunk 1): {'subject': '动物', 'predicate': '维持着', 'object': '地球的生态平衡', 'chunk': 1}
Normalized: SUB='动物', PRED='维持着', OBJ='地球的生态平衡'
Status: Kept (New Unique Triple)

--- Example 4 ---
Original Triple (Chunk 1): {'subject': '动物', 'predicate': '是', 'object': '大自然奇迹', 'chunk': 1}
Normalized: SUB='动物', PRED='是', OBJ='大自然奇迹'
Status: Kept (New Unique Triple)

--- Example 5 ---
Original Triple (Chunk 1): {'subject': '地球上的动物', 'predicate': '种类', 'object': '繁多', 'chunk': 1}
Normaliz

Let's summarize the normalization results and display the final list of unique, clean triples.

In [50]:
# --- Summary of Normalization ---
print("\n--- Normalization & De-duplication Summary ---")
print(f"Original extracted triple count: {original_count}")
print(f"Triples removed (empty/invalid components): {empty_removed_count}")
print(f"Duplicate triples removed: {duplicates_removed_count}")
final_count = len(normalized_triples)
print(f"Final unique, normalized triple count: {final_count}")
print("-" * 25)

# Display a sample of normalized triples using Pandas
print("\n--- Final Normalized Triples ---")
if normalized_triples:
    normalized_df = pd.DataFrame(normalized_triples)
    display(normalized_df)
else:
    print("No valid triples remain after normalization.")
print("-" * 25)


--- Normalization & De-duplication Summary ---
Original extracted triple count: 191
Triples removed (empty/invalid components): 0
Duplicate triples removed: 9
Final unique, normalized triple count: 182
-------------------------

--- Final Normalized Triples ---


,subject,predicate,object,source_chunk
0,动物,是,地球上生物多样性的重要组成部分,1
1,动物,与,植物、微生物共同构成复杂的生态系统,1
2,动物,维持着,地球的生态平衡,1
3,动物,是,大自然奇迹,1
4,地球上的动物,种类,繁多,1
...,...,...,...,...
177,狼,用于,研究有关行为学和群体动力学,9
178,狼,发挥,关键作用,9
179,保护狼以及它们的栖息地,变得,尤为重要,10
180,保护狼以及它们的栖息地,能维持,生态系统健康发展,10


-------------------------


## Step 9: Build the Knowledge Graph with NetworkX

Using the clean `normalized_triples`, we construct a `networkx` directed graph (`DiGraph`).
*   Subjects and Objects become nodes.
*   Predicates become edge labels.

In [51]:
# Create an empty directed graph
knowledge_graph = nx.DiGraph()

print("Initialized an empty NetworkX DiGraph.")
# Visualize the initial empty graph state
print("--- Initial Graph Info ---")
try:
    # Try the newer method first
    print(nx.info(knowledge_graph))
except AttributeError:
    # Fallback for different NetworkX versions
    print(f"Type: {type(knowledge_graph).__name__}")
    print(f"Number of nodes: {knowledge_graph.number_of_nodes()}")
    print(f"Number of edges: {knowledge_graph.number_of_edges()}")
print("-" * 25)

Initialized an empty NetworkX DiGraph.
--- Initial Graph Info ---
Type: DiGraph
Number of nodes: 0
Number of edges: 0
-------------------------


Now, we add the triples to the graph one by one, showing the graph's growth.

In [52]:
print("Adding triples to the NetworkX graph...")

added_edges_count = 0
update_interval = 5  # How often to print graph info update

if not normalized_triples:
    print("Warning: No normalized triples to add to the graph.")
else:
    for i, triple in enumerate(normalized_triples):
        subject_node = triple["subject"]
        object_node = triple["object"]
        predicate_label = triple["predicate"]

        # Nodes are added automatically when adding edges, but explicit calls are fine too
        # knowledge_graph.add_node(subject_node)
        # knowledge_graph.add_node(object_node)

        # Add the directed edge with the predicate as a 'label' attribute
        knowledge_graph.add_edge(subject_node, object_node, label=predicate_label)
        added_edges_count += 1

        # --- Visualize Graph Growth ---
        if (i + 1) % update_interval == 0 or (i + 1) == len(normalized_triples):
            print(
                f"\n--- Graph Info after adding Triple #{i + 1} --- ({subject_node} -> {object_node})"
            )
            try:
                # Try the newer method first
                print(nx.info(knowledge_graph))
            except AttributeError:
                # Fallback for different NetworkX versions
                print(f"Type: {type(knowledge_graph).__name__}")
                print(f"Number of nodes: {knowledge_graph.number_of_nodes()}")
                print(f"Number of edges: {knowledge_graph.number_of_edges()}")
            # For very large graphs, printing info too often can be slow. Adjust interval.

print(f"\nFinished adding triples. Processed {added_edges_count} edges.")

Adding triples to the NetworkX graph...

--- Graph Info after adding Triple #5 --- (地球上的动物 -> 繁多)
Type: DiGraph
Number of nodes: 7
Number of edges: 5

--- Graph Info after adding Triple #10 --- (很多动物 -> 朋友、助手或工作伙伴的角色)
Type: DiGraph
Number of nodes: 16
Number of edges: 10

--- Graph Info after adding Triple #15 --- (实验动物 -> 科学研究的重要对象)
Type: DiGraph
Number of nodes: 25
Number of edges: 15

--- Graph Info after adding Triple #20 --- (偷猎和非法贸易 -> 某些物种的生存危机)
Type: DiGraph
Number of nodes: 33
Number of edges: 19

--- Graph Info after adding Triple #25 --- (动物 -> 自然界的珍宝)
Type: DiGraph
Number of nodes: 40
Number of edges: 24

--- Graph Info after adding Triple #30 --- (猫 -> 温顺和独立性)
Type: DiGraph
Number of nodes: 47
Number of edges: 29

--- Graph Info after adding Triple #35 --- (猫的呼噜声 -> 舒缓和疗愈作用)
Type: DiGraph
Number of nodes: 53
Number of edges: 34

--- Graph Info after adding Triple #40 --- (狗 -> 各种工作)
Type: DiGraph
Number of nodes: 59
Number of edges: 39

--- Graph Info after adding Triple #

Let's look at the final graph statistics and sample nodes/edges.

In [53]:
# --- Final Graph Statistics ---
num_nodes = knowledge_graph.number_of_nodes()
num_edges = knowledge_graph.number_of_edges()

print("\n--- Final NetworkX Graph Summary ---")
print(f"Total unique nodes (entities): {num_nodes}")
print(f"Total unique edges (relationships): {num_edges}")

if num_edges != added_edges_count and isinstance(knowledge_graph, nx.DiGraph):
    print(
        f"Note: Added {added_edges_count} edges, but graph has {num_edges}. DiGraph overwrites edges with same source/target. Use MultiDiGraph if multiple edges needed."
    )

if num_nodes > 0:
    try:
        density = nx.density(knowledge_graph)
        print(f"Graph density: {density:.4f}")
        if nx.is_weakly_connected(knowledge_graph):
            print(
                "The graph is weakly connected (all nodes reachable ignoring direction)."
            )
        else:
            num_components = nx.number_weakly_connected_components(knowledge_graph)
            print(f"The graph has {num_components} weakly connected components.")
    except Exception as e:
        print(
            f"Could not calculate some graph metrics: {e}"
        )  # Handle potential errors on empty/small graphs
else:
    print("Graph is empty, cannot calculate metrics.")
print("-" * 25)

# --- Sample Nodes ---
print("\n--- Sample Nodes (First 10) ---")
if num_nodes > 0:
    nodes_sample = list(knowledge_graph.nodes())[:10]
    display(pd.DataFrame(nodes_sample, columns=["Node Sample"]))
else:
    print("Graph has no nodes.")

# --- Sample Edges ---
print("\n--- Sample Edges (First 10 with Labels) ---")
if num_edges > 0:
    edges_sample = []
    for u, v, data in list(knowledge_graph.edges(data=True))[:10]:
        edges_sample.append(
            {"Source": u, "Target": v, "Label": data.get("label", "N/A")}
        )
    display(pd.DataFrame(edges_sample))
else:
    print("Graph has no edges.")
print("-" * 25)


--- Final NetworkX Graph Summary ---
Total unique nodes (entities): 226
Total unique edges (relationships): 180
Note: Added 182 edges, but graph has 180. DiGraph overwrites edges with same source/target. Use MultiDiGraph if multiple edges needed.
Graph density: 0.0035
The graph has 47 weakly connected components.
-------------------------

--- Sample Nodes (First 10) ---


,Node Sample
0,动物
1,地球上生物多样性的重要组成部分
2,植物、微生物共同构成复杂的生态系统
3,地球的生态平衡
4,大自然奇迹
5,地球上的动物
6,繁多
7,昆虫
8,地球的每一个角落
9,大象等大型动物



--- Sample Edges (First 10 with Labels) ---


,Source,Target,Label
0,动物,地球上生物多样性的重要组成部分,是
1,动物,植物、微生物共同构成复杂的生态系统,与
2,动物,地球的生态平衡,维持着
3,动物,大自然奇迹,是
4,动物,人类有着密切的关系,与
5,动物,农业和科研中扮演着重要角色,在
6,动物,丰厚的历史和信仰,承载
7,动物,自然界的珍宝,是
8,动物,人类的朋友和老师,是
9,地球上的动物,繁多,种类


-------------------------


## Step 10: Visualize the Graph Interactively with ipycytoscape

Finally, we visualize the constructed graph interactively within the notebook using `ipycytoscape`. We'll convert the `networkx` data, define styles, and display the widget.

In [54]:
print("Preparing interactive visualization...")

# --- Check Graph Validity for Visualization ---
can_visualize = False
if "knowledge_graph" not in locals() or not isinstance(knowledge_graph, nx.Graph):
    print("Error: 'knowledge_graph' not found or is not a NetworkX graph.")
elif knowledge_graph.number_of_nodes() == 0:
    print("NetworkX Graph is empty. Cannot visualize.")
else:
    print(
        f"Graph seems valid for visualization ({knowledge_graph.number_of_nodes()} nodes, {knowledge_graph.number_of_edges()} edges)."
    )
    can_visualize = True

Preparing interactive visualization...
Graph seems valid for visualization (226 nodes, 180 edges).


### 10.1 Convert NetworkX Data to Cytoscape Format

`ipycytoscape` requires nodes and edges in a specific JSON-like format (list of dictionaries).

In [55]:
cytoscape_nodes = []
cytoscape_edges = []

if can_visualize:
    print("Converting nodes...")
    # Calculate degrees for node sizing
    node_degrees = dict(knowledge_graph.degree())
    max_degree = max(node_degrees.values()) if node_degrees else 1

    for node_id in knowledge_graph.nodes():
        degree = node_degrees.get(node_id, 0)
        # Simple scaling for node size (adjust logic as needed)
        node_size = 15 + (degree / max_degree) * 50 if max_degree > 0 else 15

        cytoscape_nodes.append(
            {
                "data": {
                    "id": str(node_id),  # ID must be string
                    "label": str(node_id).replace(
                        " ", "\n"
                    ),  # Display label (wrap spaces)
                    "degree": degree,
                    "size": node_size,
                    "tooltip_text": f"Entity: {str(node_id)}\nDegree: {degree}",  # Tooltip on hover
                }
            }
        )
    print(f"Converted {len(cytoscape_nodes)} nodes.")

    print("Converting edges...")
    edge_count = 0
    for u, v, data in knowledge_graph.edges(data=True):
        edge_id = f"edge_{edge_count}"  # Unique edge ID
        predicate_label = data.get("label", "")
        cytoscape_edges.append(
            {
                "data": {
                    "id": edge_id,
                    "source": str(u),
                    "target": str(v),
                    "label": predicate_label,  # Label on edge
                    "tooltip_text": f"Relationship: {predicate_label}",  # Tooltip on hover
                }
            }
        )
        edge_count += 1
    print(f"Converted {len(cytoscape_edges)} edges.")

    # Combine into the final structure
    cytoscape_graph_data = {"nodes": cytoscape_nodes, "edges": cytoscape_edges}

    # Visualize the converted structure (first few nodes/edges)
    print("\n--- Sample Cytoscape Node Data (First 2) ---")
    print(json.dumps(cytoscape_graph_data["nodes"][:2], indent=2))
    print("\n--- Sample Cytoscape Edge Data (First 2) ---")
    print(json.dumps(cytoscape_graph_data["edges"][:2], indent=2))
    print("-" * 25)
else:
    print("Skipping data conversion as graph is not valid for visualization.")
    cytoscape_graph_data = {"nodes": [], "edges": []}

Converting nodes...
Converted 226 nodes.
Converting edges...
Converted 180 edges.

--- Sample Cytoscape Node Data (First 2) ---
[
  {
    "data": {
      "id": "\u52a8\u7269",
      "label": "\u52a8\u7269",
      "degree": 9,
      "size": 31.071428571428573,
      "tooltip_text": "Entity: \u52a8\u7269\nDegree: 9"
    }
  },
  {
    "data": {
      "id": "\u5730\u7403\u4e0a\u751f\u7269\u591a\u6837\u6027\u7684\u91cd\u8981\u7ec4\u6210\u90e8\u5206",
      "label": "\u5730\u7403\u4e0a\u751f\u7269\u591a\u6837\u6027\u7684\u91cd\u8981\u7ec4\u6210\u90e8\u5206",
      "degree": 1,
      "size": 16.785714285714285,
      "tooltip_text": "Entity: \u5730\u7403\u4e0a\u751f\u7269\u591a\u6837\u6027\u7684\u91cd\u8981\u7ec4\u6210\u90e8\u5206\nDegree: 1"
    }
  }
]

--- Sample Cytoscape Edge Data (First 2) ---
[
  {
    "data": {
      "id": "edge_0",
      "source": "\u52a8\u7269",
      "target": "\u5730\u7403\u4e0a\u751f\u7269\u591a\u6837\u6027\u7684\u91cd\u8981\u7ec4\u6210\u90e8\u5206",
      "labe

### 10.2 Create and Configure the Cytoscape Widget

In [56]:
if can_visualize:
    print("Creating ipycytoscape widget...")
    cyto_widget = ipycytoscape.CytoscapeWidget()
    print("Widget created.")

    print("Loading graph data into widget...")
    cyto_widget.graph.add_graph_from_json(cytoscape_graph_data, directed=True)
    print("Data loaded.")
else:
    print("Skipping widget creation.")
    cyto_widget = None

Creating ipycytoscape widget...
Widget created.
Loading graph data into widget...
Data loaded.


### 10.3 Define Visual Style

We use a CSS-like syntax to control the appearance of nodes and edges.

In [57]:
if cyto_widget:
    print("Defining enhanced colorful and interactive visual style...")
    # More vibrant and colorful styling with a modern color scheme
    visual_style = [
        {
            "selector": "node",
            "style": {
                "label": "data(label)",
                "width": "data(size)",
                "height": "data(size)",
                "background-color": "#3498db",  # Bright blue
                "background-opacity": 0.9,
                "color": "#ffffff",  # White text
                "font-size": "12px",
                "font-weight": "bold",
                "text-valign": "center",
                "text-halign": "center",
                "text-wrap": "wrap",
                "text-max-width": "100px",
                "text-outline-width": 2,
                "text-outline-color": "#2980b9",  # Matching outline
                "text-outline-opacity": 0.7,
                "border-width": 3,
                "border-color": "#1abc9c",  # Turquoise border
                "border-opacity": 0.9,
                "shape": "ellipse",
                "transition-property": "background-color, border-color, border-width, width, height",
                "transition-duration": "0.3s",
                "tooltip-text": "data(tooltip_text)",
            },
        },
        {
            "selector": "node:selected",
            "style": {
                "background-color": "#e74c3c",  # Pomegranate red
                "border-width": 4,
                "border-color": "#c0392b",
                "text-outline-color": "#e74c3c",
                "width": "data(size) * 1.2",  # Enlarge selected nodes
                "height": "data(size) * 1.2",
            },
        },
        {
            "selector": "node:hover",
            "style": {
                "background-color": "#9b59b6",  # Purple on hover
                "border-width": 4,
                "border-color": "#8e44ad",
                "cursor": "pointer",
                "z-index": 999,
            },
        },
        {
            "selector": "edge",
            "style": {
                "label": "data(label)",
                "width": 2.5,
                "curve-style": "bezier",
                "line-color": "#2ecc71",  # Green
                "line-opacity": 0.8,
                "target-arrow-color": "#27ae60",
                "target-arrow-shape": "triangle",
                "arrow-scale": 1.5,
                "font-size": "10px",
                "font-weight": "normal",
                "color": "#2c3e50",
                "text-background-opacity": 0.9,
                "text-background-color": "#ecf0f1",
                "text-background-shape": "roundrectangle",
                "text-background-padding": "3px",
                "text-rotation": "autorotate",
                "edge-text-rotation": "autorotate",
                "transition-property": "line-color, width, target-arrow-color",
                "transition-duration": "0.3s",
                "tooltip-text": "data(tooltip_text)",
            },
        },
        {
            "selector": "edge:selected",
            "style": {
                "line-color": "#f39c12",  # Yellow-orange
                "target-arrow-color": "#d35400",
                "width": 4,
                "text-background-color": "#f1c40f",
                "color": "#ffffff",  # White text
                "z-index": 998,
            },
        },
        {
            "selector": "edge:hover",
            "style": {
                "line-color": "#e67e22",  # Orange on hover
                "width": 3.5,
                "cursor": "pointer",
                "target-arrow-color": "#d35400",
                "z-index": 997,
            },
        },
        {
            "selector": ".center-node",
            "style": {
                "background-color": "#16a085",  # Teal
                "background-opacity": 1,
                "border-width": 4,
                "border-color": "#1abc9c",  # Turquoise border
                "border-opacity": 1,
            },
        },
    ]

    print("Setting enhanced visual style on widget...")
    cyto_widget.set_style(visual_style)

    # Apply a better animated layout
    cyto_widget.set_layout(
        name="cose",
        nodeRepulsion=5000,
        nodeOverlap=40,
        idealEdgeLength=120,
        edgeElasticity=200,
        nestingFactor=6,
        gravity=90,
        numIter=2500,
        animate=True,
        animationDuration=1000,
        initialTemp=300,
        coolingFactor=0.95,
    )

    # Add a special class to main nodes (Marie Curie)
    if len(cyto_widget.graph.nodes) > 0:
        main_nodes = [
            node.data["id"]
            for node in cyto_widget.graph.nodes
            if node.data.get("degree", 0) > 10
        ]

        # Create gradient styles for center nodes
        for i, node_id in enumerate(main_nodes):
            # Use vibrant colors for center nodes
            center_style = {
                "selector": f'node[id = "{node_id}"]',
                "style": {
                    "background-color": "#9b59b6",  # Purple
                    "background-opacity": 0.95,
                    "border-width": 4,
                    "border-color": "#8e44ad",  # Darker purple border
                    "border-opacity": 1,
                    "text-outline-width": 3,
                    "text-outline-color": "#8e44ad",
                    "font-size": "14px",
                },
            }
            visual_style.append(center_style)

        # Update the style with the new additions
        cyto_widget.set_style(visual_style)

    print("Enhanced colorful and interactive style applied successfully.")
else:
    print("Skipping style definition.")

Defining enhanced colorful and interactive visual style...
Setting enhanced visual style on widget...
Enhanced colorful and interactive style applied successfully.


### 10.4 Set Layout Algorithm

We choose an algorithm to automatically arrange the nodes and edges.

In [58]:
if cyto_widget:
    print("Setting layout algorithm ('cose')...")
    # cose (Compound Spring Embedder) is often good for exploring connections
    cyto_widget.set_layout(
        name="cose",
        animate=True,
        # Adjust parameters for better spacing/layout
        nodeRepulsion=4000,  # Increase repulsion
        nodeOverlap=40,  # Increase overlap avoidance
        idealEdgeLength=120,  # Slightly longer ideal edges
        edgeElasticity=150,
        nestingFactor=5,
        gravity=100,  # Increase gravity slightly
        numIter=1500,  # More iterations
        initialTemp=200,
        coolingFactor=0.95,
        minTemp=1.0,
    )
    print("Layout set. The graph will arrange itself when displayed.")
else:
    print("Skipping layout setting.")

Setting layout algorithm ('cose')...
Layout set. The graph will arrange itself when displayed.


### 10.5 Display the Interactive Graph

The final step is to render the interactive widget in the notebook output below.

In [59]:
from IPython.display import display

if cyto_widget:
    print("Displaying interactive graph widget below...")
    print(
        "Interact: Zoom (scroll), Pan (drag background), Move Nodes (drag nodes), Hover for details."
    )
else:
    print("No widget to display.")

print("\n" + "-" * 25 + "\nEnd of Visualization Step." + "\n" + "-" * 25)
display(cyto_widget)

Displaying interactive graph widget below...
Interact: Zoom (scroll), Pan (drag background), Move Nodes (drag nodes), Hover for details.

-------------------------
End of Visualization Step.
-------------------------


CytoscapeWidget(cytoscape_layout={'name': 'cose', 'nodeRepulsion': 4000, 'nodeOverlap': 40, 'idealEdgeLength':…

In [60]:
for u, v, data in list(knowledge_graph.edges(data=True)):
    print(u, v, data.get("label", "N/A"))  # Print edge labels for verification

动物 地球上生物多样性的重要组成部分 是
动物 植物、微生物共同构成复杂的生态系统 与
动物 地球的生态平衡 维持着
动物 大自然奇迹 是
动物 人类有着密切的关系 与
动物 农业和科研中扮演着重要角色 在
动物 丰厚的历史和信仰 承载
动物 自然界的珍宝 是
动物 人类的朋友和老师 是
地球上的动物 繁多 种类
昆虫 地球的每一个角落 占据了
大象等大型动物 体型和力量令人惊叹 以
每种动物 适应其生存环境的独特特征 通过进化发展出
很多动物 朋友、助手或工作伙伴的角色 扮演着
宠物 人类无尽的陪伴和情感支持 给予
一些工作动物 人类完成某些特殊任务 帮助
家畜和家禽 人类提供食物来源 为
实验动物 科学研究的重要对象 是
人类活动 动物的栖息地 威胁着
人类活动 狼的生存环境面临诸多挑战 导致
工业化进程、城市化扩张、森林砍伐 许多物种濒临灭绝 导致
偷猎和非法贸易 某些物种的生存危机 加剧
各国政府和国际组织 动物保护法律法规 加强
各国政府和国际组织 拯救濒危动物的努力 推动
许多文化 神灵的化身或图腾 视为
动物角色 人类无法用语言表达的情感和哲理 用来表现
保护动物及其生存环境 保护我们生态系统 是
保护动物及其生存环境 保护人类自身的未来 是
猫 一种常见的宠物 是
猫 温顺和独立性 闻名于
猫 柔软的毛发 有
猫 敏锐的感官 有
猫 敏捷的动作 有
猫 很好地适应各种居住环境 可以
猫的呼噜声 舒缓和疗愈作用 被认为具有
狗 人类的宠物 是
狗 人类最好的朋友之一 是
狗 陪伴 能够提供
狗 保护家庭 能够
狗 各种工作 可以进行
狗 很多品种 有
狗 重要角色 扮演
狐狸 犬科动物 属于
狐狸 聪明,狡猾 闻名
狐狸 尖尖的鼻子和耳朵 具有
狐狸 小型哺乳动物、鸟类以及昆虫为食 以
狐狸 植物果实 吃
狐狸 独居动物 是
狐狸 其他狐狸 交往
狐狸 自己的领地 建立
狐狸 善于变化和狡黠的化身 被看作
狐狸 不同角色 被赋予
狐狸 聪明形象 被刻画成
狐狸 智谋 运用
狐狸 诡计 通过
狐狸 害兽 被视为
狐狸 重要角色 扮演
狐狸 啮齿动物数量 帮助控制
狐狸 食物链一部分 成为
狐狸 生态系统重要组成部分 是
狐狸 保护生态平衡重要对象 成为
狐狸 极具适应能力的动物 是
狐狸 自然界特殊存在 成为
狐狸 具有性格化特征的动物 成为
狐狸 自然

## Step 11: Conclusion and Next Steps

We have now walked through a very granular process:
1.  Setup libraries and LLM connection.
2.  Defined and chunked input text, visualizing intermediate steps.
3.  Defined detailed prompts for the LLM.
4.  Iterated through chunks (demonstrated with one), showing raw LLM output, parsed JSON, and extracted triples for each.
5.  Aggregated, normalized, and de-duplicated triples, showing the results.
6.  Built the `networkx` graph step-by-step, showing its growth.
7.  Converted data for `ipycytoscape` and visualized the final interactive knowledge graph directly in the notebook.

This detailed breakdown should make the transformation from unstructured text to a structured, visual knowledge graph much clearer.

**Potential Improvements and Further Exploration:**
*   **Run Full Loop:** Execute the LLM extraction and normalization across *all* chunks for a complete graph.
*   **Advanced Normalization:** Implement entity linking or relationship clustering.
*   **Error Handling:** Add retries for LLM calls, better handling of persistent chunk failures.
*   **Prompt Tuning:** Experiment with different models, prompts, and parameters.
*   **Evaluation:** Assess the quality of extracted triples (Precision/Recall).
*   **Richer Visualization:** Use node types for colors/shapes, add community detection coloring, implement more interactive features using ipycytoscape callbacks.
*   **Graph Analysis:** Apply `networkx` algorithms (centrality, paths, etc.).
*   **Persistence:** Store results in a graph database (Neo4j, etc.).